# Две дополнительные модели

Для сравнения с CatBoost обучаем две альтернативы: Logistic Regression и RandomForest

In [6]:
from __future__ import annotations

import hashlib
import json
import platform
import sys
from datetime import datetime
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RANDOM_STATE = 42
ROOT_DIR = Path.cwd()
if ROOT_DIR.name == 'notebooks':
    ROOT_DIR = ROOT_DIR.parent

DATA_PATH = ROOT_DIR / 'data' / 'loan_data.csv'
ARTIFACT_DIR = ROOT_DIR / 'artifacts' / 'exp3'
MODEL_DIR = ARTIFACT_DIR / 'models'
TRACKING_PATH = ARTIFACT_DIR / 'experiments.jsonl'

for path in [ARTIFACT_DIR, MODEL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 140)
plt.style.use('default')


def file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def load_data():
    df = pd.read_csv(DATA_PATH)
    target = 'loan_status'
    X = df.drop(columns=[target])
    y = df[target].astype(int)
    numeric_features = X.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()
    return df, X, y, numeric_features, categorical_features


def append_jsonl(path: Path, row: dict):
    with path.open('a', encoding='utf-8') as f:
        f.write(json.dumps(row, ensure_ascii=False, default=str) + '\n')


def log_experiment(name: str, model_family: str, params: dict, val_metrics: dict, test_metrics: dict, artifacts: dict | None = None):
    row = {
        'run_id': hashlib.md5(f'{name}-{datetime.utcnow().isoformat()}'.encode()).hexdigest()[:12],
        'created_at': datetime.utcnow().isoformat(),
        'name': name,
        'model_family': model_family,
        'data_path': str(DATA_PATH.relative_to(ROOT_DIR)),
        'data_sha256': file_sha256(DATA_PATH),
        'python': sys.version.split()[0],
        'platform': platform.platform(),
        'target': 'loan_status',
        'params': params,
        'val_metrics': val_metrics,
        'test_metrics': test_metrics,
        'artifacts': artifacts or {},
    }
    append_jsonl(TRACKING_PATH, row)
    return row


from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split


def make_splits(X, y):
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=0.25, random_state=RANDOM_STATE, stratify=y_train_val
    )
    return X_train, X_val, X_test, y_train, y_val, y_test, X_train_val, y_train_val


def predict_scores(model, X_data):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X_data)[:, 1]
    return model.predict(X_data)


def compute_metrics(model, X_data, y_true) -> dict[str, float]:
    y_pred = model.predict(X_data)
    y_score = predict_scores(model, X_data)
    return {
        'roc_auc': float(roc_auc_score(y_true, y_score)),
        'average_precision': float(average_precision_score(y_true, y_score)),
        'f1': float(f1_score(y_true, y_pred)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred)),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
    }

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

In [7]:
df, X, y, numeric_features, categorical_features = load_data()
X_train, X_val, X_test, y_train, y_val, y_test, X_train_val, y_train_val = make_splits(X, y)
print('train/val/test:', X_train.shape, X_val.shape, X_test.shape)

train/val/test: (27000, 13) (9000, 13) (9000, 13)


In [8]:
preprocess_linear = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', make_one_hot_encoder())]), categorical_features),
])
preprocess_tree = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), numeric_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', make_one_hot_encoder())]), categorical_features),
])

In [10]:
logreg = Pipeline([
    ('preprocess', preprocess_linear),
    ('model', LogisticRegression(max_iter=2000, C=1.0, random_state=RANDOM_STATE)),
])
logreg.fit(X_train, y_train)
logreg_val = compute_metrics(logreg, X_val, y_val)
logreg_test = compute_metrics(logreg, X_test, y_test)
logreg_path = MODEL_DIR / 'logistic_regression.joblib'
joblib.dump(logreg, logreg_path)
log_experiment(
    name='logistic_regression',
    model_family='sklearn',
    params={'C': 1.0, 'max_iter': 2000},
    val_metrics=logreg_val,
    test_metrics=logreg_test,
    artifacts={'model': str(logreg_path.relative_to(ROOT_DIR))},
)
print('validation:', logreg_val)
print('test:', logreg_test)

validation: {'roc_auc': 0.9509917857142857, 'average_precision': 0.8507915213185508, 'f1': 0.7439339184305628, 'precision': 0.7689434364994664, 'recall': 0.7205, 'accuracy': 0.8897777777777778, 'balanced_accuracy': 0.8293214285714285}
test: {'roc_auc': 0.956113642857143, 'average_precision': 0.8632157422186292, 'f1': 0.7677286742034943, 'precision': 0.7896405919661733, 'recall': 0.747, 'accuracy': 0.8995555555555556, 'balanced_accuracy': 0.8450714285714286}


/var/folders/r2/wy5cx1bj3hd5n3q1hwfbtq6r0000gn/T/ipykernel_11791/2112990490.py:58: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'run_id': hashlib.md5(f'{name}-{datetime.utcnow().isoformat()}'.encode()).hexdigest()[:12],
/var/folders/r2/wy5cx1bj3hd5n3q1hwfbtq6r0000gn/T/ipykernel_11791/2112990490.py:59: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'created_at': datetime.utcnow().isoformat(),


In [11]:
random_forest = Pipeline([
    ('preprocess', preprocess_tree),
    ('model', RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=5,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight='balanced_subsample',
    )),
])
random_forest.fit(X_train, y_train)
rf_val = compute_metrics(random_forest, X_val, y_val)
rf_test = compute_metrics(random_forest, X_test, y_test)
rf_path = MODEL_DIR / 'random_forest_simple.joblib'
joblib.dump(random_forest, rf_path)
log_experiment(
    name='random_forest_simple',
    model_family='sklearn',
    params={'n_estimators': 200, 'min_samples_leaf': 5, 'class_weight': 'balanced_subsample'},
    val_metrics=rf_val,
    test_metrics=rf_test,
    artifacts={'model': str(rf_path.relative_to(ROOT_DIR))},
)
print('validation:', rf_val)
print('test:', rf_test)

validation: {'roc_auc': 0.9705972857142858, 'average_precision': 0.918291829745069, 'f1': 0.8121851763012713, 'precision': 0.7805440295066851, 'recall': 0.8465, 'accuracy': 0.913, 'balanced_accuracy': 0.8892500000000001}
test: {'roc_auc': 0.9735967857142857, 'average_precision': 0.9270735839838956, 'f1': 0.8301526717557252, 'precision': 0.7937956204379562, 'recall': 0.87, 'accuracy': 0.9208888888888889, 'balanced_accuracy': 0.9027142857142857}


/var/folders/r2/wy5cx1bj3hd5n3q1hwfbtq6r0000gn/T/ipykernel_11791/2112990490.py:58: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'run_id': hashlib.md5(f'{name}-{datetime.utcnow().isoformat()}'.encode()).hexdigest()[:12],
/var/folders/r2/wy5cx1bj3hd5n3q1hwfbtq6r0000gn/T/ipykernel_11791/2112990490.py:59: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'created_at': datetime.utcnow().isoformat(),
